[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/paper_implementation/blob/main/mean_flow_sanity_check_fashion_mnist.ipynb)

# Conditional MeanFlow — FashionMNIST + DiT + Muon

`mean_flow_sanity_check.ipynb`에서 **dataset만 MNIST → FashionMNIST**로 바꾼 버전이다.
모델, MeanFlow/JVP 식, Muon+AdamW 구성, `norm_eps=0.01`, 20,000-step 학습 설정은 그대로 유지한다.
코드는 한 줄 압축을 풀었고, 학습 전에 dataset preview와 TensorBoard computation graph를 확인한다.


## 0. Setup


In [ ]:
!pip -q install datasets tensorboard

import math
import os
import random
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from torch.func import jvp
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets as tv_datasets
from torchvision import transforms
from torchvision.transforms import ToTensor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Use a Colab GPU runtime.")

DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.mha.set_fastpath_enabled(False)

TRAIN_STEPS = 20_000
BATCH_SIZE = 128
DIAG_BATCH = 64
MUON_LR = 2e-2
ADAMW_LR = 3e-4
MUON_MOMENTUM = 0.95
P_MEAN = -0.4
P_STD = 1.0
DATA_PROPORTION = 0.75
NORM_EPS = 0.01
NUM_CLASSES = 10
CONDITION_POOL = tuple(range(NUM_CLASSES))
SAMPLE_EVERY = 1000
DIAG_EVERY = 250
LOG_EVERY = 50

ROOT = "/content/meanflow_fashion_mnist_dit_sanity"
RUN = "fashion_mnist_cond_muon20k_" + time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(ROOT, RUN)
SAMPLE_DIR = os.path.join(RUN_DIR, "samples")
CKPT_DIR = os.path.join(RUN_DIR, "checkpoints")
LOG_DIR = os.path.join(ROOT, "tensorboard", RUN)

for path in (SAMPLE_DIR, CKPT_DIR, LOG_DIR):
    os.makedirs(path, exist_ok=True)

writer = SummaryWriter(LOG_DIR)


## 1. FashionMNIST data

원본 MNIST 전처리와 동일하게 `28×28` 이미지를 사방 2 pixel padding해 `32×32`로 만들고 `[0,1] → [-1,1]`로 변환한다.
Hugging Face를 우선 사용하고 실패하면 torchvision으로 fallback한다.


In [ ]:
FASHION_CLASS_NAMES = (
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
)

to_tensor = ToTensor()


def transform_image(pil_image):
    image = to_tensor(pil_image)
    image = F.pad(
        image,
        (2, 2, 2, 2),
        value=0.0,
    )
    return image * 2.0 - 1.0


def collate_hf(batch):
    images = torch.stack(
        [transform_image(item["image"]) for item in batch]
    )
    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long,
    )
    return images, labels


try:
    try:
        dataset = load_dataset("zalando-datasets/fashion_mnist")
        source_name = "zalando-datasets/fashion_mnist"
    except Exception:
        dataset = load_dataset("anonyme449/fashion_mnist")
        source_name = "anonyme449/fashion_mnist"

    train_loader = DataLoader(
        dataset["train"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True,
        collate_fn=collate_hf,
    )
    test_loader = DataLoader(
        dataset["test"],
        batch_size=DIAG_BATCH,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=collate_hf,
    )
    print("FashionMNIST: Hugging Face", source_name)

except Exception as error:
    print("HF failed, torchvision fallback:", repr(error))

    fallback_transform = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Pad(2),
            transforms.Normalize((0.5,), (0.5,)),
        ]
    )
    train_dataset = tv_datasets.FashionMNIST(
        "/content/fashion_mnist_data",
        train=True,
        transform=fallback_transform,
        download=True,
    )
    test_dataset = tv_datasets.FashionMNIST(
        "/content/fashion_mnist_data",
        train=False,
        transform=fallback_transform,
        download=True,
    )
    train_loader = DataLoader(
           train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=DIAG_BATCH,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    print("FashionMNIST: torchvision")


### 1-1. Training 전에 data 확인


In [ ]:
preview_images, preview_labels = next(iter(train_loader))

print("images:", tuple(preview_images.shape))
print("range:", float(preview_images.min()), float(preview_images.max()))
print("labels:", tuple(preview_labels.shape))

figure, axes = plt.subplots(4, 4, figsize=(7, 7))

for index, axis in enumerate(axes.flat):
    image = preview_images[index, 0].add(1.0).div(2.0)
    label = int(preview_labels[index])

    axis.imshow(
        image,
        cmap="gray",
        vmin=0.0,
        vmax=1.0,
    )
    axis.set_title(
        f"{label}: {FASHION_CLASS_NAMES[label]}",
        fontsize=8,
    )
    axis.axis("off")

figure.tight_layout()
plt.show()


## 2. Conditional DiT


In [ ]:
class ScalarEmbed(nn.Module):

    def __init__(self, dim, fourier_dim=128):
        super().__init__()
        self.fourier_dim = fourier_dim
        self.mlp = nn.Sequential(
            nn.Linear(fourier_dim, dim),
            nn.SiLU(),
            nn.Linear(dim, dim),
        )

    def forward(self, scalar):
        half = self.fourier_dim // 2
        frequencies = torch.exp(
            -math.log(10000.0)
            * torch.arange(
                half,
                device=scalar.device,
                dtype=scalar.dtype,
            )
            / half
        )
        phase = scalar[:, None] * frequencies[None, :] * 2.0 * math.pi
        embedding = torch.cat(
            [phase.cos(), phase.sin()],
            dim=-1,
        )
        return self.mlp(embedding)


class Block(nn.Module):

    def __init__(self, dim=224, heads=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(
            dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.norm2 = nn.LayerNorm(
            dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.attn = nn.MultiheadAttention(
            dim,
            heads,
            batch_first=True,
        )
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * dim, dim),
        )
        self.mod = nn.Sequential(
            nn.SiLU(),
            nn.Linear(dim, 6 * dim),
        )
        nn.init.zeros_(self.mod[-1].weight)
        nn.init.zeros_(self.mod[-1].bias)

    def forward(self, tokens, conditioning):
        (
            shift1,
            scale1,
            gate1,
            shift2,
            scale2,
            gate2,
        ) = self.mod(conditioning).chunk(6, dim=-1)

        hidden = self.norm1(tokens)
        hidden = hidden * (1.0 + scale1[:, None, :]) + shift1[:, None, :]
        attended, _ = self.attn(
            hidden,
            hidden,
            hidden,
            need_weights=True,
        )
        tokens = tokens + gate1[:, None, :] * attended

        hidden = self.norm2(tokens)
        hidden = hidden * (1.0 + scale2[:, None, :]) + shift2[:, None, :]
        return tokens + gate2[:, None, :] * self.mlp(hidden)


class CondDiT(nn.Module):

    def __init__(self, dim=224, depth=4, patch=4):
        super().__init__()
        self.patch = patch
        self.pe = nn.Conv2d(
            1,
            dim,
            patch,
            patch,
        )
        self.pos = nn.Parameter(
            torch.zeros(1, 64, dim)
        )
        self.te = ScalarEmbed(dim)
        self.he = ScalarEmbed(dim)
        self.ye = nn.Embedding(NUM_CLASSES, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, 8) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(
            dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.fmod = nn.Sequential(
            nn.SiLU(),
            nn.Linear(dim, 2 * dim),
        )
        self.out = nn.Linear(
            dim,
            patch * patch,
        )
        nn.init.normal_(self.pos, std=0.02)
        nn.init.normal_(self.ye.weight, std=0.02)
        nn.init.zeros_(self.fmod[-1].weight)
        nn.init.zeros_(self.fmod[-1].bias)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, images, t, interval, labels):
        batch_size = images.shape[0]

        tokens = self.pe(images)
        tokens = tokens.flatten(2).transpose(1, 2)
        tokens = tokens + self.pos

        conditioning = (
            self.te(t)
            + self.he(interval)
            + self.ye(labels)
        )
        for block in self.blocks:
            tokens = block(tokens, conditioning)

        shift, scale = self.fmod(conditioning).chunk(2, dim=-1)
        tokens = self.norm(tokens)
        tokens = tokens * (1.0 + scale[:, None, :]) + shift[:, None, :]
        patches = self.out(tokens)
        patches = patches.view(
            batch_size,
            8,
            8,
            self.patch,
            self.patch,
            1,
        )
        images_out = torch.einsum(
            "nhwpqc->nchpwq",
            patches,
        )
        return images_out.reshape(
            batch_size,
            1,
            32,
            32,
        )


model = CondDiT().to(DEVICE)
print(model)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)
print("params M:", parameter_count / 1e6)


### 2-1. TensorBoard computation graph

학습 전에 dummy batch forward를 확인하고 TensorBoard의 **Graphs** 탭에 computation graph를 기록한다.


In [ ]:
graph_images, graph_labels = next(iter(train_loader))
graph_images = graph_images[:4].to(DEVICE)
graph_labels = graph_labels[:4].to(DEVICE)

graph_t = torch.full(
    (4,),
    0.75,
    device=DEVICE,
)
graph_interval = torch.full(
    (4,),
    0.50,
    device=DEVICE,
)

model.eval()

with torch.no_grad():
    graph_output = model(
        graph_images,
        graph_t,
        graph_interval,
        graph_labels,
    )

print("graph input:", tuple(graph_images.shape))
print("graph output:", tuple(graph_output.shape))

try:
    writer.add_graph(
        model,
        (
            graph_images,
            graph_t,
            graph_interval,
            graph_labels,
        ),
    )
    writer.flush()
    print("graph written:", LOG_DIR)
except Exception as error:
    warnings.warn("add_graph failed: " + repr(error))

model.train()


In [ ]:
%load_ext tensorboard
%tensorboard --logdir $LOG_DIR


## 3. MeanFlow objective + Muon hybrid


In [ ]:
def logit_normal(sample_count):
    normal = torch.randn(
        sample_count,
        device=DEVICE,
    )
    return torch.sigmoid(
        normal * P_STD + P_MEAN
    )


def sample_tuple(images):
    first = logit_normal(len(images))
    second = logit_normal(len(images))
    t = torch.maximum(first, second)
    r = torch.minimum(first, second)

    border_count = int(
        len(images) * DATA_PROPORTION
    )
    r[:border_count] = t[:border_count]

    noise = torch.randn_like(images)
    t_image = t[:, None, None, None]
    z_t = (1.0 - t_image) * images + t_image * noise
    velocity = noise - images

    return z_t, velocity, r, t


def mf_outputs(z_t, velocity, r, t, labels):

    def meanflow_fn(z_arg, t_arg, r_arg):
        return model(
            z_arg,
            t_arg,
            t_arg - r_arg,
            labels,
        )

    average_velocity, total_derivative = jvp(
        meanflow_fn,
        (z_t, t, r),
        (
            velocity,
            torch.ones_like(t),
            torch.zeros_like(r),
        ),
    )
    interval = (t - r)[:, None, None, None]
    target = (
        velocity
        - interval * total_derivative
    ).detach()

    return average_velocity, target


def loss_fn(average_velocity, target):
    sse = (
        average_velocity - target
    ).pow(2).flatten(1).sum(1)
    weight = (
        sse.detach() + NORM_EPS
    ).reciprocal()

    return (
        (sse * weight).mean(),
        sse.mean(),
    )


In [ ]:
@torch.no_grad()
def ns5(gradient, steps=5, eps=1e-7):
    matrix = gradient.float()
    transpose_back = matrix.shape[0] > matrix.shape[1]

    if transpose_back:
        matrix = matrix.T

    matrix = matrix / (matrix.norm() + eps)

    a = 3.4445
    b = -4.775
    c = 2.0315

    for _ in range(steps):
        gram = matrix @ matrix.T
        matrix = (
            a * matrix
            + (b * gram + c * gram @ gram) @ matrix
        )

    if transpose_back:
        matrix = matrix.T

    return matrix.to(gradient.dtype)


class MuonFallback(torch.optim.Optimizer):

    def __init__(self, params, lr=0.02, momentum=0.95):
        super().__init__(
            params,
            {
                "lr": lr,
                "momentum": momentum,
            },
        )

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for parameter in group["params"]:
                if parameter.grad is None:
                    continue

                state = self.state[parameter]
                buffer = state.setdefault(
                    "momentum_buffer",
                    torch.zeros_like(parameter.grad),
                )
                buffer.mul_(
                    group["momentum"]
                ).add_(
                    parameter.grad
                )

                update = ns5(
                    parameter.grad
                    + group["momentum"] * buffer
                )
                update *= math.sqrt(
                    max(
                        1.0,
                        parameter.shape[0] / parameter.shape[1],
                    )
                )
                parameter.add_(
                    update,
                    alpha=-group["lr"],
                )


muon_params = []
adam_params = []

for name, parameter in model.named_parameters():
    if (
        parameter.ndim == 2
        and (
            ".attn." in name
            or ".mlp." in name
        )
    ):
        muon_params.append(parameter)
    else:
        adam_params.append(parameter)

if hasattr(torch.optim, "Muon"):
    opt_muon = torch.optim.Muon(
        muon_params,
        lr=MUON_LR,
        momentum=MUON_MOMENTUM,
        weight_decay=0.0,
    )
    backend = "torch.optim.Muon"
else:
    opt_muon = MuonFallback(
        muon_params,
        lr=MUON_LR,
        momentum=MUON_MOMENTUM,
    )
    backend = "fallback Muon"

opt_adam = torch.optim.AdamW(
    adam_params,
    lr=ADAMW_LR,
    betas=(0.9, 0.99),
    eps=1e-8,
    weight_decay=0.0,
)

print("Muon backend:", backend)
print("Muon tensors:", len(muon_params))
print("AdamW tensors:", len(adam_params))


## 4. Sampling / diagnostics


In [ ]:
fixed_noise = torch.randn(
    16,
    1,
    32,
    32,
    device=DEVICE,
)
sample_rng = random.Random(SEED + 10000)

diag_images, diag_labels = next(iter(test_loader))
diag_images = diag_images.to(DEVICE)
diag_labels = diag_labels.to(DEVICE)
diag_noise = torch.randn_like(diag_images)

first = logit_normal(len(diag_images))
second = logit_normal(len(diag_images))
diag_t = torch.maximum(first, second)
diag_r = torch.minimum(first, second)

border_count = int(
    len(diag_images) * DATA_PROPORTION
)
diag_r[:border_count] = diag_t[:border_count]


@torch.no_grad()
def save_samples(step):
    model.eval()

    sampled_labels = [
        sample_rng.choice(CONDITION_POOL)
        for _ in range(16)
    ]
    labels = torch.tensor(
        sampled_labels,
        device=DEVICE,
    )
    one = torch.ones(
        16,
        device=DEVICE,
    )

    generated = (
        fixed_noise
        - model(
            fixed_noise,
            one,
            one,
            labels,
        )
    )
    generated = (
        generated.clamp(-1.0, 1.0)
        .add(1.0)
        .div(2.0)
    )

    figure, axes = plt.subplots(
        4,
        4,
        figsize=(7, 7),
    )

    for index, axis in enumerate(axes.flat):
        label = sampled_labels[index]
        axis.imshow(
            generated[index, 0].cpu(),
            cmap="gray",
            vmin=0.0,
            vmax=1.0,
        )
        axis.set_title(
            f"y={label} {FASHION_CLASS_NAMES[label]}",
            fontsize=8,
        )
        axis.axis("off")

    figure.suptitle(
        f"Conditional MeanFlow step {step}"
    )
    figure.tight_layout()
    figure.savefig(
        os.path.join(
            SAMPLE_DIR,
            f"step_{step:05d}.png",
        ),
        dpi=160,
        bbox_inches="tight",
    )
    plt.close(figure)
    model.train()


@torch.no_grad()
def diagnostics():
    model.eval()

    t_image = diag_t[:, None, None, None]
    z_t = (
        (1.0 - t_image) * diag_images
        + t_image * diag_noise
    )
    velocity = diag_noise - diag_images

    average_velocity, target = mf_outputs(
        z_t,
        velocity,
        diag_r,
        diag_t,
        diag_labels,
    )

    mse = (
        average_velocity - target
    ).pow(2).flatten(1).mean(1)
    cosine = F.cosine_similarity(
        average_velocity.flatten(1),
        target.flatten(1),
        dim=1,
    )

    interval_mask = diag_r < diag_t
    boundary_mask = diag_r == diag_t

    model.train()

    return (
        mse.mean().item(),
        mse[interval_mask].mean().item(),
        cosine[interval_mask].mean().item(),
        mse[boundary_mask].mean().item(),
    )


def grad_norm():
    squared_norm = 0.0

    for parameter in model.parameters():
        if parameter.grad is not None:
            squared_norm += (
                parameter.grad.detach()
                .float()
                .pow(2)
                .sum()
                .item()
            )

    return math.sqrt(squared_norm)


def save_ckpt(step):
    torch.save(
        {
            "step": step,
            "model": model.state_dict(),
            "muon": opt_muon.state_dict(),
            "adamw": opt_adam.state_dict(),
            "config": {
                "muon_lr": MUON_LR,
                "adamw_lr": ADAMW_LR,
                "norm_eps": NORM_EPS,
                "condition_pool": CONDITION_POOL,
            },
        },
        os.path.join(
            CKPT_DIR,
            f"step_{step:05d}.pt",
        ),
    )


## 5. Training — 20,000 steps


In [ ]:
model.train()
train_iterator = iter(train_loader)
start_time = time.time()

save_samples(0)

for step in range(1, TRAIN_STEPS + 1):
    try:
        images, labels = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        images, labels = next(train_iterator)

    images = images.to(
        DEVICE,
        non_blocking=True,
    )
    labels = labels.to(
        DEVICE,
        non_blocking=True,
    )

    z_t, velocity, r, t = sample_tuple(images)
    average_velocity, target = mf_outputs(
        z_t,
        velocity,
        r,
        t,
        labels,
    )
    loss, raw_sse = loss_fn(
        average_velocity,
        target,
    )

    opt_muon.zero_grad(set_to_none=True)
    opt_adam.zero_grad(set_to_none=True)

    loss.backward()
    gradient_norm = grad_norm()

    opt_muon.step()
    opt_adam.step()

    if step % LOG_EVERY == 0:
        writer.add_scalar(
            "train/loss_adaptive",
            loss.item(),
            step,
        )
        writer.add_scalar(
            "train/raw_sse",
            raw_sse.item(),
            step,
        )
        writer.add_scalar(
            "train/grad_norm",
            gradient_norm,
            step,
        )
        writer.add_scalar(
            "run/elapsed_minutes",
            (time.time() - start_time) / 60.0,
            step,
        )

    if step % DIAG_EVERY == 0:
        (
            mse_all,
            mse_interval,
            cosine_interval,
            mse_boundary,
        ) = diagnostics()

        writer.add_scalar(
            "diagnostic/raw_mse_all",
            mse_all,
            step,
        )
        writer.add_scalar(
            "diagnostic/raw_mse_interval",
            mse_interval,
            step,
        )
        writer.add_scalar(
            "diagnostic/interval_cosine",
            cosine_interval,
            step,
        )
        writer.add_scalar(
            "diagnostic/boundary_mse",
            mse_boundary,
            step,
        )

        print(
            f"step={step:05d} "
            f"loss={loss.item():.6f} "
            f"grad={gradient_norm:.4f} "
            f"mse={mse_all:.4f} "
            f"interval_cos={cosine_interval:.4f} "
            f"boundary={mse_boundary:.4f}"
        )

    if step % SAMPLE_EVERY == 0:
        save_samples(step)
        save_ckpt(step)

    if step % 250 == 0:
        writer.flush()

writer.flush()

final_path = os.path.join(
    RUN_DIR,
    "meanflow_cond_dit_muon_fashion_mnist_20k.pt",
)
torch.save(
    {
        "step": TRAIN_STEPS,
        "model": model.state_dict(),
    },
    final_path,
)
writer.close()

print("saved:", RUN_DIR)


## 6. TensorBoard training curves

학습 후 같은 log directory에서 Graphs와 scalar curve를 함께 확인한다.


In [ ]:
%reload_ext tensorboard
%tensorboard --logdir $LOG_DIR
